# 🚀 Amazon ML Challenge 2026 — Google Colab Complete Execution Pipeline

This notebook provides a complete, robust walkthrough to run the entire Entity Resolution pipeline on Google Colab with GPU acceleration and full RAM utilization.

### 📌 What This Notebook Does:
1. **Hardware Inspection**: Verifies T4/A100 GPU and max memory availability.
2. **Google Drive Integration**: Mounts Drive so your outputs and checkpoints are automatically preserved even if Colab disconnects.
3. **Git LFS Cloning**: Pulls the repository and all large LFS artifacts (the 2.3 GB candidate index, normalized datasets, and 343k+ blocking progress).
4. **Environment Setup**: Installs all ML libraries (`sentence-transformers`, `peft`, `xgboost`, etc.).
5. **Execution**:
   - **Track A (Fast Baseline Submission)**: Finishes Stage 1 blocking and trains the lexical XGBoost model to lock in a baseline submission immediately.
   - **Track B (Full GPU Deep Learning Pipeline)**: Trains BGE-M3 LoRA, extracts dense embeddings with GPU, and builds the full ensemble.
6. **Export & Download**: Packages the outputs into a `.zip` file for instant download or saves directly to Google Drive.

## ⚙️ Step 1: System Hardware & GPU Check
Make sure your Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU` or `A100 GPU`).

In [ ]:
# Check GPU
!nvidia-smi

# Check CPU and RAM details
import psutil, os
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
cpus = os.cpu_count()
print(f"\n[HARDWARE] Allocated CPU Cores: {cpus} | System RAM: {ram_gb:.2f} GB")

## 💾 Step 2: (Recommended) Mount Google Drive
Mounting Google Drive allows you to back up your `output/` directory directly to Drive so you never lose progress.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive successfully mounted to /content/drive")

## 📥 Step 3: Clone Repository & Pull Git LFS Artifacts
If your repo is private, provide your GitHub Personal Access Token (PAT) when prompted.

In [ ]:
# Ensure Git LFS is installed
!apt-get install -y git-lfs > /dev/null
!git lfs install

import os
# Repository configuration
REPO_URL = "https://github.com/priyanshu-ogdev/AmazonMLChallenge2026.git"

if not os.path.exists("AmazonMLChallenge2026"):
    print("Cloning repository...")
    !git clone $REPO_URL
    %cd AmazonMLChallenge2026
else:
    %cd AmazonMLChallenge2026
    !git pull origin main

# Pull all LFS tracked artifacts (2.3GB index, candidate pairs, datasets)
print("Pulling Git LFS artifacts...")
!git lfs pull
print("Git LFS pull complete!")

## 📦 Step 4: Install Dependencies & Setup Environment

In [ ]:
# Install python dependencies
!pip install -q -r code/business_entity_resolution/requirements.txt

# Make all shell scripts executable
!chmod +x scripts/*.sh

print("Environment initialized!")

## 🧪 Step 5: Run Pre-Flight Environment Smoke Test
This executes the test suite to verify that PyTorch with CUDA, XGBoost, and sentence-transformers are 100% operational.

In [ ]:
!bash scripts/00_verify_environment.sh --python-path python3

## 🏃 Step 6: Execute the Pipeline

Select your preferred execution track below:

### ⚡ Track 1: Fast Baseline Submission
Finishes Stage 1 blocking, skips dense neural embedding extraction (`--baseline-only`), trains the lexical XGBoost model, and creates a valid `submission.csv` in record time.

In [ ]:
# Run Fast Baseline Pipeline
!bash scripts/run_all_phases.sh --run-mode Full --skip-gpu --baseline-only --python-path python3

### 🔥 Track 2: Full GPU End-to-End Pipeline (Post-Baseline Upgrade)
Trains the BGE-M3 LoRA adapter with GPU acceleration, extracts dense embeddings, trains the full calibrated XGBoost model, and creates the upgraded submission.

In [ ]:
# Run Full GPU Pipeline
# !bash scripts/run_all_phases.sh --run-mode Full --python-path python3

## 📊 Step 7: Validate Submission Format

In [ ]:
!bash scripts/05_validate_submission.sh --python-path python3

## 🎁 Step 8: Package & Download Outputs

Save your entire `output/` directory as a zip and copy it directly to your Google Drive and local machine.

In [ ]:
import os
from google.colab import files

# 1. Download submission.csv directly
sub_file = "output/phase4_predictions/submission.csv"
if os.path.exists(sub_file):
    print("Downloading final submission.csv...")
    files.download(sub_file)
else:
    print("Submission file not found yet.")

# 2. Zip the output directory (excluding temporary worker scratch chunks)
print("Packaging output/ into output_artifacts.zip...")
!zip -r -q output_artifacts.zip output/ -x "output/**/_tmp_worker_chunks/*" "output/**/*.tmp"

# 3. Backup output_artifacts.zip to Google Drive
drive_backup_dir = "/content/drive/MyDrive/AmazonMLChallenge_Backup"
os.makedirs(drive_backup_dir, exist_ok=True)
!cp output_artifacts.zip "{drive_backup_dir}/output_artifacts.zip"
print(f"Backed up output_artifacts.zip to Google Drive at {drive_backup_dir}!")

# 4. (Optional) Download the zip to your local PC
# files.download('output_artifacts.zip')